In [4]:
# Step 0: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, PowerTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, roc_auc_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

# -----------------------------
# Step 1: Load Data
columns = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent','hot',
    'num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count','srv_count',
    'serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate',
    'dst_host_count','dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate',
    'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty_level'
]

file_train_path = "../data/KDDTrain+.txt"
file_test_path = "../data/KDDTest+.txt"

train_df = pd.read_csv(file_train_path, header=None, names=columns)
test_df = pd.read_csv(file_test_path, header=None, names=columns)

# -----------------------------
# Step 2: Data Cleaning
train_df.dropna(inplace=True)
test_df.dropna(inplace=True)

train_df.drop_duplicates(inplace=True)
test_df.drop_duplicates(inplace=True)

# -----------------------------
# Step 3: Binary Target
train_df['attack_binary'] = train_df['label'].apply(lambda x: 0 if x == 'normal' else 1)
test_df['attack_binary'] = test_df['label'].apply(lambda x: 0 if x == 'normal' else 1)

# -----------------------------
# Step 4: Multi-class Target
train_df['attack_multi'] = train_df['label']
test_df['attack_multi'] = test_df['label']

# -----------------------------
# Step 5: Encode categorical features
categorical_cols = ['protocol_type', 'service', 'flag']
train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

# Align test columns with train
missing_cols = set(train_df.columns) - set(test_df.columns)
for col in missing_cols:
    test_df[col] = 0
test_df = test_df[train_df.columns]

# -----------------------------
# Step 6: Scale numeric features
exclude_cols = ['label', 'difficulty_level', 'attack_binary', 'attack_multi']
numeric_cols = train_df.select_dtypes(include=np.number).columns.difference(exclude_cols)

scaler = MinMaxScaler()
train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])

# -----------------------------
# Step 7: Prepare Features and Targets

# Binary
X_train_bin = train_df.drop(columns=exclude_cols)
y_train_bin = train_df['attack_binary']
X_test_bin = test_df.drop(columns=exclude_cols)
y_test_bin = test_df['attack_binary']

# Multi-class
X_train_multi = X_train_bin.copy()
y_train_multi = train_df['attack_multi']
X_test_multi = X_test_bin.copy()
y_test_multi = test_df['attack_multi']

# -----------------------------
# Step 8: PCA + PowerTransformer
pca = PCA(n_components=15, random_state=42)
X_train_pca = pca.fit_transform(X_train_bin)
X_test_pca = pca.transform(X_test_bin)

pt = PowerTransformer(method='yeo-johnson')
X_train_power = pt.fit_transform(X_train_pca)
X_test_power = pt.transform(X_test_pca)

# -----------------------------
# Step 9: KMeans clustering (2 clusters for binary consistency)
kmeans = KMeans(n_clusters=2, random_state=42)
train_clusters = kmeans.fit_predict(X_train_power)
test_clusters = kmeans.predict(X_test_power)

X_train_power = np.concatenate([X_train_power, train_clusters.reshape(-1,1)], axis=1)
X_test_power = np.concatenate([X_test_power, test_clusters.reshape(-1,1)], axis=1)

# -----------------------------
# Step 10: Handle imbalance

# Binary - RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_train_bin_bal, y_train_bin_bal = rus.fit_resample(X_train_power, y_train_bin)

# Multi-class - SMOTE (k_neighbors=1 to handle rare classes)
smote_multi = SMOTE(sampling_strategy='not majority', random_state=42, k_neighbors=1)
X_train_multi_bal, y_train_multi_bal = smote_multi.fit_resample(X_train_power, y_train_multi)

# -----------------------------
# Step 11: Define Models

binary_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

multi_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, multi_class='auto'),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# -----------------------------
# Step 12: Train and evaluate Binary Classification
results_bin = {}
for name, model in binary_models.items():
    print(f"\nTraining {name} (Binary)...")
    model.fit(X_train_bin_bal, y_train_bin_bal)
    y_pred_bin = model.predict(X_test_power)
    acc_bin = accuracy_score(y_test_bin, y_pred_bin)
    results_bin[name] = acc_bin
    print(f"{name} Accuracy: {acc_bin*100:.2f}%")
    print(classification_report(y_test_bin, y_pred_bin, target_names=["Normal", "Attack"]))

# -----------------------------
# Step 13: Train and evaluate Multi-Class Classification
results_multi = {}
for name, model in multi_models.items():
    print(f"\nTraining {name} (Multi-Class)...")
    model.fit(X_train_multi_bal, y_train_multi_bal)
    y_pred_multi = model.predict(X_test_power)
    acc_multi = accuracy_score(y_test_multi, y_pred_multi)
    results_multi[name] = acc_multi
    print(f"{name} Multi-Class Accuracy: {acc_multi*100:.2f}%")
    print(classification_report(y_test_multi, y_pred_multi, zero_division=0))



Training Logistic Regression (Binary)...
Logistic Regression Accuracy: 76.47%
              precision    recall  f1-score   support

      Normal       0.66      0.92      0.77      9711
      Attack       0.92      0.64      0.76     12833

    accuracy                           0.76     22544
   macro avg       0.79      0.78      0.76     22544
weighted avg       0.81      0.76      0.76     22544


Training Decision Tree (Binary)...
Decision Tree Accuracy: 80.20%
              precision    recall  f1-score   support

      Normal       0.70      0.93      0.80      9711
      Attack       0.93      0.70      0.80     12833

    accuracy                           0.80     22544
   macro avg       0.82      0.82      0.80     22544
weighted avg       0.83      0.80      0.80     22544


Training Random Forest (Binary)...
Random Forest Accuracy: 80.39%
              precision    recall  f1-score   support

      Normal       0.69      0.97      0.81      9711
      Attack       0.97 

d:\.Upasana\Springboard Main\SentinelNet\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
d:\.Upasana\Springboard Main\SentinelNet\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression Multi-Class Accuracy: 64.72%
                 precision    recall  f1-score   support

        apache2       0.00      0.00      0.00       737
           back       0.19      0.66      0.29       359
buffer_overflow       0.00      0.00      0.00        20
      ftp_write       0.00      0.00      0.00         3
   guess_passwd       0.60      0.00      0.00      1231
     httptunnel       0.00      0.00      0.00       133
           imap       0.00      0.00      0.00         1
        ipsweep       0.46      0.97      0.63       141
           land       0.18      0.57      0.28         7
     loadmodule       0.00      0.00      0.00         2
       mailbomb       0.00      0.00      0.00       293
          mscan       0.00      0.00      0.00       996
       multihop       0.07      0.17      0.10        18
          named       0.00      0.00      0.00        17
        neptune       0.92      0.98      0.95      4657
           nmap       0.27      0.90  

KeyboardInterrupt: 